<a href="https://colab.research.google.com/github/aquilino/A-Simple-printf/blob/master/IA_esp32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np

In [15]:
# Datos sintéticos: [feature1, feature2, feature3], etiquetas: 0=feliz, 1=triste, 2=enfadado
X_train = np.array([
    # Feliz (movimiento alto, frecuencia media, variabilidad baja)
    [0.8, 0.5, 0.2], [0.7, 0.6, 0.1], [0.9, 0.4, 0.3],

    # Triste (movimiento bajo, frecuencia baja, variabilidad baja)
    [0.1, 0.2, 0.1], [0.2, 0.1, 0.0], [0.3, 0.3, 0.2],

    # Enfadado (movimiento medio, frecuencia alta, variabilidad alta)
    [0.5, 0.9, 0.8], [0.4, 0.8, 0.7], [0.6, 0.7, 0.9]
], dtype=np.float32)

y_train = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=np.int32)

In [16]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(3,)),  # Capa oculta
    tf.keras.layers.Dense(3, activation='softmax')  # Salida: 3 emociones
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("comenzando entrenamiento....")
model.fit(X_train, y_train, epochs=100)
print("Modelo entrenado...")

comenzando entrenamiento....
Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 1.0890
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3333 - loss: 1.0878
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.3333 - loss: 1.0865
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.3333 - loss: 1.0852
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.3333 - loss: 1.0840
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.3333 - loss: 1.0827
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.3333 - loss: 1.0815
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3333 - loss: 1.0802
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3333 - loss: 1.0790
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3333 - loss: 1.0777
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.3333 - loss: 1.0765
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3333 - loss: 1.0753

In [17]:
import tensorflow as tf
import numpy as np

# Assume 'model' is your trained model from the previous code

# Sample input data representing a happy emotion
sample_input = np.array([[0.1, 0.8, 0.1]], dtype=np.float32)

# Make the prediction
prediction = model.predict(sample_input)

# Get the predicted class (emotion)
predicted_class = np.argmax(prediction)

# Interpret the prediction
emotion_labels = {0: "feliz", 1: "triste", 2: "enfadado"}
predicted_emotion = emotion_labels[predicted_class]

# Print the results
print("Sample Input:", sample_input)
print("Prediction (Raw Output):", prediction)
print("Predicted Class:", predicted_class)
print("Predicted Emotion:", predicted_emotion)

# Convertir a TFLite con cuantización
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

# Guardar el modelo quantizado
with open('emotion_model_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Sample Input: [[0.1 0.8 0.1]]
Prediction (Raw Output): [[0.27310783 0.3360424  0.39084983]]
Predicted Class: 2
Predicted Emotion: enfadado
Saved artifact at '/tmp/tmp2fari0i4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139646094866384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094868112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094867152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094866576: TensorSpec(shape=(), dtype=tf.resource, name=None)


# Nueva sección

In [18]:
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Probar con un dato de ejemplo
input_data = np.array([[0.5, 0.9, 0.8]], dtype=np.float32)  # Enfadado
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])
print("Predicción:", np.argmax(output_data))  # Debería imprimir "2"

Predicción: 2


In [19]:
# @title Texto de título predeterminado
# Convertir a TFLite sin cuantización (para simplicidad)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Guardar el modelo
with open('emotion_model.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpkpdk40_5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139646094866384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094868112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094867152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094866576: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [20]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open('emotion_model_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)

Saved artifact at '/tmp/tmp06r54w75'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139646094866384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094868112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094867152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139646094866576: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [21]:
!xxd -i emotion_model_quant.tflite > model.h